# MTSamples — EDA

~5,000 de-identified medical transcription samples, scraped from mtsamples.com. Loaded from a public GitHub CSV mirror — no Kaggle account needed. Columns (per the source): `description`, `medical_specialty`, `sample_name`, `transcription`, `keywords`.



In [ ]:
!pip install -q textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 18.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import textstat

pd.set_option("display.max_colwidth", 80)

CSV_URL = "https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv"
df = pd.read_csv(CSV_URL)

print(df.shape)
print(df.columns.tolist())
df.head()

(4999, 6)
['Unnamed: 0', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with complaint of allergies.,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female presents with complaint of aller...","allergy / immunology, allergic rhinitis, allergies, asthma, nasal sprays, rh..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with ai...","bariatrics, laparoscopic gastric bypass, weight loss programs, gastric bypas..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC today. He is a very pleasant ...","bariatrics, laparoscopic gastric bypass, heart attacks, body weight, pulmona..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement with left atrial diameter of 4.7 ...","cardiovascular / pulmonary, 2-d m-mode, doppler, aortic valve, atrial enlarg..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall thickness appear normal. The ...,"cardiovascular / pulmonary, 2-d, doppler, echocardiogram, annular, aortic ro..."


## 1. Missingness & basic shape

In [ ]:
print(df.isna().sum())
df.describe(include="all").T

Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,4999.0,NaN,NaN,NaN,2499.0,1443.231328,0.0,1249.5,2499.0,3748.5,4998.0
description,4999,2348,An example/template for a routine normal male physical exam.,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
medical_specialty,4999,40,Surgery,1103,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sample_name,4999,2377,Lumbar Discogram,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transcription,4966,2357,"PREOPERATIVE DIAGNOSIS: , Low back pain.,POSTOPERATIVE DIAGNOSIS: , Low back...",5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keywords,3931,3849,,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Specialty distribution
How many samples per medical specialty — tells you which specialties are over/under-represented if you're sampling from this for testing.

In [ ]:
specialty_counts = df["medical_specialty"].value_counts()
specialty_counts.to_frame("n_samples")

,n_samples
medical_specialty,
Surgery,1103
Consult - History and Phy.,516
Cardiovascular / Pulmonary,372
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,230
Neurology,223
SOAP / Chart / Progress Notes,166


## 3. Transcription length

In [ ]:
df["word_count"] = df["transcription"].fillna("").apply(lambda t: len(str(t).split()))
df["char_count"] = df["transcription"].fillna("").apply(lambda t: len(str(t)))

df[["word_count", "char_count"]].describe()

,word_count,char_count
count,4999.000000,4999.000000
mean,462.376275,3032.165033
std,317.585206,2002.802464
min,0.000000,0.000000
25%,239.000000,1590.500000
50%,397.000000,2659.000000
75%,614.000000,3995.000000
max,3029.000000,18425.000000


## 4. Length by specialty
Do certain specialties produce systematically longer/shorter transcriptions?

In [ ]:
df.groupby("medical_specialty")["word_count"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)

,mean,median,count
medical_specialty,,,
Autopsy,1184.250000,1448.0,8
IME-QME-Work Comp etc.,1088.625000,1129.5,16
Chiropractic,929.000000,653.0,14
Psychiatry / Psychology,810.792453,661.0,53
Hospice - Palliative Care,676.166667,687.5,6
Endocrinology,637.842105,621.0,19
Neurosurgery,576.946809,497.5,94
Orthopedic,559.123944,503.0,355
Consult - History and Phy.,558.063953,489.0,516


## 5. Readability baseline
Flesch-Kincaid grade level and Flesch Reading Ease, computed per transcription. This is the number you'd want to see drop after your Simplifier runs on these — i.e. this table is your 'before' baseline.

In [ ]:
def safe_readability(text):
    text = str(text)
    if len(text.split()) < 10:
        return pd.Series({"flesch_reading_ease": None, "flesch_kincaid_grade": None})
    return pd.Series({
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
    })

readability = df["transcription"].fillna("").apply(safe_readability)
df = pd.concat([df, readability], axis=1)

df[["flesch_reading_ease", "flesch_kincaid_grade"]].describe()

,flesch_reading_ease,flesch_kincaid_grade
count,4926.000000,4926.000000
mean,44.536200,10.106561
std,12.871420,1.948147
min,-91.002045,4.713351
25%,38.494768,8.845590
50%,45.771909,10.016683
75%,52.375643,11.131395
max,81.809811,27.318636


## 6. Readability by specialty

In [ ]:
df.groupby("medical_specialty")[["flesch_kincaid_grade", "flesch_reading_ease"]].mean().sort_values("flesch_kincaid_grade", ascending=False)

,flesch_kincaid_grade,flesch_reading_ease
medical_specialty,,
Speech - Language,12.592154,37.322779
Sleep Medicine,11.721307,34.231653
Lab Medicine - Pathology,10.873930,37.477734
Physical Medicine - Rehab,10.864060,40.757017
Radiology,10.846476,37.058801
Cosmetic / Plastic Surgery,10.833718,44.989916
Pain Management,10.733997,42.005280
ENT - Otolaryngology,10.709349,41.758239
Endocrinology,10.562120,44.755111
